# Semantic Chunking – BSI Umsetzungshinweise Cloud-Dienste

This notebook loads pre-analyzed Document Intelligence JSON, performs **semantic chunking** by section headings,
generates embeddings with `text-embedding-3-large`, creates an Azure AI Search index, and pushes the chunks.

In [2]:
import json
import os

import tiktoken
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
)
from azure.search.documents.models import VectorizedQuery
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

# ── Configuration ──────────────────────────────────────────────────────────
SEARCH_SERVICE_NAME = "pocaisearchbam"
SEARCH_ENDPOINT = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
INDEX_NAME = "push-semantic-chunking-1"

AI_FOUNDRY_ENDPOINT = "https://poc-sweden-ai-plattform.services.ai.azure.com/"
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_DIMENSIONS = 3072

DATA_FILE = os.path.join(
    os.getcwd(),
    "data",
    "GNB B 147_2001 Rev. 1_layout.pdf.json",
)

# ── Authentication ─────────────────────────────────────────────────────────
openai_client = AzureOpenAI(
    api_version="2024-02-01",
    azure_endpoint=AI_FOUNDRY_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

search_credential = AzureKeyCredential(os.environ["AI_SEARCH_KEY"])

print("Configuration loaded.")


Configuration loaded.


## 1 — Load Document Intelligence JSON

In [3]:
with open(DATA_FILE, "r", encoding="utf-8") as f:
    doc_intel_output = json.load(f)

result = doc_intel_output["analyzeResult"]

print(f"API version : {result['apiVersion']}")
print(f"Model       : {result['modelId']}")
print(f"Pages       : {len(result['pages'])}")
print(f"Paragraphs  : {len(result['paragraphs'])}")
print(f"Content len : {len(result['content'])} chars")

API version : 2024-11-30
Model       : prebuilt-layout
Pages       : 46
Paragraphs  : 1485
Content len : 60754 chars


## 2 — Extract metadata (title, headers, footers)

In [4]:
SKIP_ROLES = {"pageHeader", "pageFooter", "pageNumber", "footnote"}

# Extract the document title (first paragraph with role 'title')
title = None
for p in result["paragraphs"]:
    if p.get("role") == "title":
        title = p["content"]
        break

print(f"Title: {title}")

# Quick overview of roles in the document
from collections import Counter

role_counts = Counter(p.get("role", "<body>") for p in result["paragraphs"])
for role, count in role_counts.most_common():
    print(f"  {role:20s} {count}")

Title: Typ B(U)F-Versandstück Transport- und Lagerbehälter CASTOR® X/A17 Thermische Auslegung
  <body>               1320
  pageNumber           65
  pageHeader           46
  sectionHeading       44
  pageFooter           8
  title                1
  footnote             1


## 3 — Semantic chunking by section headings

In [5]:
def extract_chunks(result):
    """Split paragraphs into chunks at sectionHeading / title boundaries.

    Returns a list of dicts:
        {"chunk_id": int, "heading": str, "text": str}

    Paragraphs with roles in SKIP_ROLES are excluded from the chunk text.
    """
    chunks = []
    current_heading = None
    current_paragraphs = []
    chunk_id = 1

    for paragraph in result["paragraphs"]:
        role = paragraph.get("role")

        # Skip noise paragraphs
        if role in SKIP_ROLES:
            continue

        # New section boundary
        if role in ("sectionHeading", "title"):
            # Save the previous section
            if current_heading is not None:
                chunks.append({
                    "chunk_id": chunk_id,
                    "heading": current_heading,
                    "text": " ".join(current_paragraphs),
                })
                chunk_id += 1
            current_heading = paragraph["content"]
            current_paragraphs = []
        else:
            current_paragraphs.append(paragraph["content"])

    # Don't forget the last section
    if current_heading is not None:
        chunks.append({
            "chunk_id": chunk_id,
            "heading": current_heading,
            "text": " ".join(current_paragraphs),
        })

    return chunks


chunks = extract_chunks(result)
print(f"Total chunks: {len(chunks)}\n")

for c in chunks[:8]:
    preview = c["text"][:120] + "..." if len(c["text"]) > 120 else c["text"]
    print(f"[{c['chunk_id']:>3}] {c['heading']}")
    print(f"      {preview}\n")

Total chunks: 45

[  1] Typ B(U)F-Versandstück Transport- und Lagerbehälter CASTOR® X/A17 Thermische Auslegung
      Bericht-Nr: GNB B 147/2001 Datum (Revision 1): Juni 2003 Seitenzahl: 46 Rev. 1 Name Unterschrift Datum Ersteller Vallent...

[  2] Revisionsstand
      Revision Datum Ersteller Grund für Änderungen 0 Juni 2003 R. Vallentin, WTI Dr. A. Leber, WTI Erstausgabe 1 Juni 2003 R....

[  3] Inhaltsverzeichnis
      Seite Revisionsstand 2 Inhaltsverzeichnis 3 Tabellenverzeichnis 5 Abbildungsverzeichnis 6 1. Einleitung und Zusammenfass...

[  4] Tabellenverzeichnis
      Seite Tabelle 1: Hauptabmessungen der Verpackung und der Transporthaube 11 Tabelle 2: Werkstoffkennwerte 15 Tabelle 3: V...

[  5] Abbildungsverzeichnis
      Seite Abbildung 1: Axialmodell mit Hauptabmessungen 12 Abbildung 2: Radialmodell des Behälters mit Tragkorb und Brennele...

[  6] 1. Einleitung und Zusammenfassung
      Der Transport- und Lagerbehälter CASTOR® X/A17 ist für den Transport von bis zu 17 abgeb

## 4 — Generate embeddings

In [6]:
MAX_TOKENS = 8191
_enc = tiktoken.get_encoding("cl100k_base")


def get_embedding(text: str) -> list[float]:
    """Get embedding vector, truncating to MAX_TOKENS if needed."""
    tokens = _enc.encode(text)
    if len(tokens) > MAX_TOKENS:
        text = _enc.decode(tokens[:MAX_TOKENS])
    response = openai_client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return response.data[0].embedding


for i, chunk in enumerate(chunks):
    embed_input = chunk["heading"] + " " + chunk["text"]
    chunk["vector"] = get_embedding(embed_input)
    print(f"  Embedded {i + 1}/{len(chunks)}: [{chunk['chunk_id']}] {chunk['heading'][:60]}")

print(f"\nDone. Vector dimension sample: {len(chunks[0]['vector'])}")

  Embedded 1/45: [1] Typ B(U)F-Versandstück Transport- und Lagerbehälter CASTOR® 
  Embedded 2/45: [2] Revisionsstand
  Embedded 3/45: [3] Inhaltsverzeichnis
  Embedded 4/45: [4] Tabellenverzeichnis
  Embedded 5/45: [5] Abbildungsverzeichnis
  Embedded 6/45: [6] 1. Einleitung und Zusammenfassung
  Embedded 7/45: [7] 2. Verkehrsrechtliche Anforderungen
  Embedded 8/45: [8] 3. Randbedingungen
  Embedded 9/45: [9] 3.1 Wärmeabfuhrmechanismen
  Embedded 10/45: [10] 3.2 Abmessungen
  Embedded 11/45: [11] 3.3 Werkstoffkennwerte
  Embedded 12/45: [12] 3.4 Brennelemente und Wärmeleistung
  Embedded 13/45: [13] 3.5 Beförderungsbedingungen
  Embedded 14/45: [14] 4. Berechnungsverfahren
  Embedded 15/45: [15] 5. Temperaturen unter Routine-Beförderungsbedingungen
  Embedded 16/45: [16] 5.1 Transporthaube und Luft unter der Transporthaube
  Embedded 17/45: [17] 5.1.1 Berechnungsmodell und Randbedingungen
  Embedded 18/45: [18] 5.1.2 Berechnungsergebnisse für die Transporthaube und die L
  Embedded 1

## 5 — Create the Azure AI Search index

In [7]:
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
    ),
    SearchableField(
        name="title",
        type=SearchFieldDataType.String,
        analyzer_name="de.lucene",
    ),
    SearchableField(
        name="section_heading",
        type=SearchFieldDataType.String,
        analyzer_name="de.lucene",
    ),
    SearchableField(
        name="chunk",
        type=SearchFieldDataType.String,
        analyzer_name="de.lucene",
    ),
    SimpleField(
        name="source_file",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True,
    ),
    SearchField(
        name="chunkVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="my-vector-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="my-hnsw")],
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-hnsw",
        )
    ],
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="section_heading"),
        content_fields=[SemanticField(field_name="chunk")],
    ),
)

semantic_search = SemanticSearch(configurations=[semantic_config])

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential)

# Delete existing index if present
try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception:
    pass

result_idx = index_client.create_or_update_index(index)
print(f"Index '{result_idx.name}' created.")

Deleted existing index 'push-semantic-chunking-1'.
Index 'push-semantic-chunking-1' created.


## 6 — Push documents into the index

In [8]:
source_file = os.path.basename(DATA_FILE)

documents = []
for chunk in chunks:
    documents.append({
        "id": str(chunk["chunk_id"]),
        "title": title,
        "section_heading": chunk["heading"],
        "chunk": chunk["text"],
        "source_file": source_file,
        "chunkVector": chunk["vector"],
    })

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_credential
)

BATCH_SIZE = 100
for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i : i + BATCH_SIZE]
    upload_result = search_client.upload_documents(documents=batch)
    succeeded = sum(1 for r in upload_result if r.succeeded)
    failed = sum(1 for r in upload_result if not r.succeeded)
    print(f"Batch {i // BATCH_SIZE + 1}: {succeeded} succeeded, {failed} failed")
    if failed:
        for r in upload_result:
            if not r.succeeded:
                print(f"  FAILED: {r.key} – {r.error_message}")

print(f"\nUpload complete. Total documents: {len(documents)}")

Batch 1: 45 succeeded, 0 failed

Upload complete. Total documents: 45


## 7 — Verify: query the index

In [9]:
# Quick count check
results = search_client.search(
    search_text="*",
    select=["id", "section_heading"],
    include_total_count=True,
)
print(f"Total documents in index: {results.get_count()}\n")
for r in results:
    print(f"  [{r['id']:>3}] {r['section_heading']}")

Total documents in index: 45

  [ 41] 1. Änderung (vgl. Kapitel 7.1):
  [  7] 2. Verkehrsrechtliche Anforderungen
  [ 42] 2. Änderung (vgl. Kapitel 7.1):
  [  5] Abbildungsverzeichnis
  [ 13] 3.5 Beförderungsbedingungen
  [ 24] 5.2.3 Berechnungsergebnisse für den Behälter und das Inventar
  [ 43] 3. Änderung (vgl. Kapitel 7.1):
  [ 44] Literaturverzeichnis1
  [ 22] 5.2.1 Axialmodell
  [ 26] 6.1 Berechnungsmodelle
  [ 23] 5.2.2 Radialmodell
  [ 39] 3. Änderung (vgl. Kapitel 7.1):
  [  2] Revisionsstand
  [ 14] 4. Berechnungsverfahren
  [ 15] 5. Temperaturen unter Routine-Beförderungsbedingungen
  [ 21] - Radialmodell
  [ 34] 2. Änderung:
  [ 20] - Axialmodell
  [ 30] 6.3.2 Radialmodell
  [  3] Inhaltsverzeichnis
  [  4] Tabellenverzeichnis
  [ 10] 3.2 Abmessungen
  [ 12] 3.4 Brennelemente und Wärmeleistung
  [ 19] 5.2 Behälter und Inventar
  [ 28] 6.3 Berechnungsergebnisse
  [ 40] 7.3 Ergebnisbewertung für die Erhitzungsprüfung
  [  1] Typ B(U)F-Versandstück Transport- und Lagerbehälter

In [10]:
# Hybrid search (text + vector) with semantic reranking
QUESTION = "Welche Materialien werden in diesem Dokument behandelt?"

query_vector = get_embedding(QUESTION)

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="chunkVector",
)

results = search_client.search(
    search_text=QUESTION,
    vector_queries=[vector_query],
    query_type="semantic",
    semantic_configuration_name="my-semantic-config",
    select=["id", "section_heading", "chunk"],
    top=5,
)

print(f"Query: {QUESTION}")
print("-" * 80)
for r in results:
    score = r.get("@search.reranker_score", r.get("@search.score", "N/A"))
    print(f"\n  [{r['id']:>3}] {r['section_heading']}  (score: {score})")
    preview = r["chunk"][:200] + "..." if len(r["chunk"]) > 200 else r["chunk"]
    print(f"       {preview}")

Query: Welche Materialien werden in diesem Dokument behandelt?
--------------------------------------------------------------------------------

  [ 11] 3.3 Werkstoffkennwerte  (score: 2.0638930797576904)
       Die für die thermischen Berechnungen verwendeten relevanten, temperaturabhängi- gen Werkstoffkennwerte (Wärmeleitfähigkeiten, Dichten, Wärmekapazitäten und Emissionskoeffizienten) sind in Tabelle 2 zu...

  [ 45] Literaturverzeichnis (Fortsetzung)  (score: 1.9654392004013062)
       I [9] Advisory Material for the IAEA Regulations for the Safe Transport of Radioactive Material IAEA Safety Standard Series No. TS-G-1.1 (ST2) IAEA, Vienna (2002) [10] GNS B 136/92, Rev. 1 Bestimmung ...

  [ 13] 3.5 Beförderungsbedingungen  (score: 1.8534270524978638)
       In diesem Bericht werden die folgenden drei Beförderungsbedingungen gemäß [1] behandelt. (i) Routine-Beförderungsbedingungen Unter Routine-Beförderungsbedingungen erfolgt die Beförderung des Versand- ...

  [  4] Tabellenverzei